# Classify Job Postings

Run `3-scrape-job-postings.ipynb` first so that `jobs/1-scraped_jobs.jsonl` exists.

This notebook reads the scraped jobs, uses an LLM to classify whether each role is truly an AI engineering role, and writes **all** classified jobs (including non-AI engineering ones) to `jobs/2-classified-jobs.jsonl`. Subsequent notebooks filter by `is_ai_engineering_role` to work with only the accepted roles.

In [12]:
import json
from dotenv import load_dotenv
from pathlib import Path

import pandas as pd
from openai import OpenAI
from IPython.display import HTML, display

In [13]:
load_dotenv(override=True)

True

### Load scraped jobs from file

In [14]:
scraped_jobs_path = Path("jobs") / "1-scraped_jobs.jsonl"

if not scraped_jobs_path.exists():
    raise FileNotFoundError(
        f"Scraped jobs JSONL file not found: {scraped_jobs_path.resolve()}. Run 3-analyse-aie-job-postings.ipynb first."
    )

if scraped_jobs_path.stat().st_size == 0:
    raise ValueError(
        "The scraped jobs JSONL file is empty. Run 3-analyse-aie-job-postings.ipynb first."
    )

jobs_df = pd.read_json(scraped_jobs_path, lines=True)

print(f"Scraped jobs JSONL file loaded successfully with {len(jobs_df)} entries!")

Scraped jobs JSONL file loaded successfully with 56 entries!


In [15]:
client = OpenAI()

# Alternatively, if you want to use a local model 👇

# OLLAMA_BASE_URL = "http://localhost:11434/v1"

# client = OpenAI(
#    base_url=OLLAMA_BASE_URL,
#    api_key="ollama",
# )

### Define the prompt

We define the instructions that tell the model what counts as an AI engineering role.

In [16]:
instructions = """
You classify whether a job posting is truly for an AI Engineering role.

AI Engineering definition:
- AI engineering means building applications on top of foundation models or in other words integrating them into products.
- Traditional ML engineering focuses on building, training, or tuning models; AI engineering primarily leverages existing models.
- MLOps and platform engineering are not AI engineering, as they focus on infrastructure and tooling rather than building AI-powered features.

Decision rules:
- Set is_ai_engineering_role to true when the main responsibility is building product or application features on top of foundation models or LLMs.
- Set is_ai_engineering_role to false when the role is mainly traditional software engineering, data science, analytics, ML research, model training, classical ML engineering, MLOps or platform work, or something else where AI application work is not the core responsibility.
- If the posting is ambiguous or unclear, set is_ai_engineering_role to false.
- In reason, briefly explain the main evidence for the decision in one sentence.
""".strip()

### Define the schema

We tell the model to return structured output with a boolean decision and a short reason.
https://json-schema.org/

In [17]:
schema = {
    "type": "object",
    "properties": {
        "is_ai_engineering_role": {"type": "boolean"},
        "reason": {"type": "string"},
    },
    "required": ["is_ai_engineering_role", "reason"],
    "additionalProperties": False,
}

### Make the LLM calls

We now ask the model to classify each job posting.

In [18]:
results = []

for i, (_, job) in enumerate(jobs_df.iterrows(), start=1):
    title = job["title"]
    description = job["description"]

    print(f"Screening job {i}/{len(jobs_df)}: {title}")

    response = client.responses.create(
        model="gpt-5.4-mini",  # or local model like "gemma4:e4b"
        instructions=instructions,
        input=f"""Classify this job posting.\n\nTitle: {title}\n\nDescription:\n{description}""",
        text={
            "format": {
                "type": "json_schema",
                "name": "ai_engineering_job_screening",
                "schema": schema,
                "strict": True,
            },
            "verbosity": "low",
        },
    )

    classification = json.loads(response.output_text)

    results.append(
        {
            "is_ai_engineering_role": classification["is_ai_engineering_role"],
            "llm_reason": classification["reason"],
        }
    )

Screening job 1/56: AI Engineer
Screening job 2/56: Software Engineer (Applied AI)
Screening job 3/56: Lead Software Engineer - Cloud/Python/AI Engineer
Screening job 4/56: Sr Lead Software Engineer - Cloud / ML / GenAI
Screening job 5/56: Applied AI ML Engineer Associate
Screening job 6/56: Manager, AI Engineer
Screening job 7/56: Senior Associate, AI Engineer
Screening job 8/56: Director, AI Engineer
Screening job 9/56: AI Engineer
Screening job 10/56: C3 AI Engineer
Screening job 11/56: AI Engineer
Screening job 12/56: AWS and AI Engineer
Screening job 13/56: SAP NS2 AI Data Engineer Intern
Screening job 14/56: Principal AI Engineer – Vivado EDA Tools
Screening job 15/56: AI Engineer - GenAI, Creative-X
Screening job 16/56: Senior Software Development Engineer, GenAIUX
Screening job 17/56: Advisory AI Engineer, AI Orchestration
Screening job 18/56: Early Careers: Intern, AI Prompt Engineer
Screening job 19/56: Founding AI Engineer ($130k-$200k + Equity) at Fast-growing Healthtech AI

### Check the classification summary

Before we combine the results with the scraped jobs, let's check how many jobs the model accepted and rejected.

In [19]:
results_df = pd.DataFrame(results)

ai_engineer_count = int(results_df["is_ai_engineering_role"].sum())
non_ai_engineer_count = int((~results_df["is_ai_engineering_role"]).sum())

print(f"Jobs screened by LLM: {len(results_df)}")
print(f"Jobs classified as AI engineering roles: {ai_engineer_count}")
print(f"Jobs classified as not AI engineering or unclear: {non_ai_engineer_count}")

Jobs screened by LLM: 56
Jobs classified as AI engineering roles: 42
Jobs classified as not AI engineering or unclear: 14


### Combine the results

We join the model output back to the scraped jobs.

In [20]:
screened_jobs = pd.concat([jobs_df, results_df], axis=1)

### Save the classified jobs

We save all classified jobs, including the jobs that were rejected as not AI engineering roles.

In [21]:
classified_jobs_path = Path("jobs") / "2-classified-jobs.jsonl"

screened_jobs.to_json(
    classified_jobs_path, orient="records", lines=True, force_ascii=False
)

print(f"Saved classified jobs to: {classified_jobs_path.resolve()}")

Saved classified jobs to: /Users/lukaslechner/PythonProjects/AI-Engineering-Foundations-Labs/1-what-is-ai-engineering/jobs/2-classified-jobs.jsonl


### Display the results

In [22]:
results_to_show = screened_jobs[
    ["title", "is_ai_engineering_role", "llm_reason", "job_url"]
].copy()
results_to_show["job_url"] = results_to_show["job_url"].apply(
    lambda url: f'<a href="{url}" target="_blank">{url}</a>' if pd.notna(url) else ""
)

display(HTML(results_to_show.to_html(escape=False, index=False)))

title,is_ai_engineering_role,llm_reason,job_url
AI Engineer,True,"The role centers on designing, prototyping, integrating, and evaluating LLM-based applications, RAG, agentic workflows, and AI toolchains, which matches AI engineering rather than model training or platform work.",https://www.indeed.com/viewjob?jk=913e8fdfc6701dbd
Software Engineer (Applied AI),True,"The role’s core responsibility is building product features on top of LLMs and other foundation models—multi-agent systems, LLM pipelines, evals, retrieval, and AI-powered product intelligence—rather than model research, pure ML, or infrastructure work.",https://www.indeed.com/viewjob?jk=4fce256c0b4f3824
Lead Software Engineer - Cloud/Python/AI Engineer,False,"This is primarily a software engineering/cloud/infrastructure role, with AI only listed as a preferred exposure (RAG/agents) rather than the core responsibility of building AI-powered product features.",https://www.indeed.com/viewjob?jk=c12dc46a15d43150
Sr Lead Software Engineer - Cloud / ML / GenAI,True,"The role centers on building and productionizing ML/LLM-powered product solutions with prompt engineering, evaluations, guardrails, and deployment, which matches AI engineering on top of foundation models.",https://www.indeed.com/viewjob?jk=0d7ca804d26c179d
Applied AI ML Engineer Associate,True,"The role centers on building product features with LLMs, AI agents, and RAG integrations, plus production code for scalable real-world applications, which fits AI engineering.",https://www.indeed.com/viewjob?jk=e12cc8d04f0894e6
"Manager, AI Engineer",True,"The role centers on designing and developing AI applications and enterprise solutions using LLMs, generative AI, cloud AI services, and integrations into business systems, which matches AI engineering.",https://www.indeed.com/viewjob?jk=0f3f577ed004a6af
"Senior Associate, AI Engineer",True,"The role primarily builds GenAI/LLM applications, conversational agents, and integrates foundation models into enterprise products, which fits AI engineering.",https://www.indeed.com/viewjob?jk=b660dacf14134ef6
"Director, AI Engineer",True,"The role centers on designing and delivering AI-powered products and multi-agent conversational systems on top of cloud AI services and enterprise applications, which fits AI engineering.",https://www.indeed.com/viewjob?jk=0068737a4e48dae1
AI Engineer,True,"The role is primarily about building AI-powered tools and automation workflows integrated into internal business systems, which matches AI engineering focused on applying foundation models in products.",https://www.indeed.com/viewjob?jk=b515489c29ca0dd2
C3 AI Engineer,True,"The role is centered on building production AI applications on top of existing models/platforms (LLM-powered apps, RAG pipelines, AI features, and enterprise integrations), which matches AI engineering.",https://www.indeed.com/viewjob?jk=4a04a8eb4f9b7b02
